In [11]:
import os 
import anthropic


In [12]:
key = os.environ.get("ANTHROPIC_API_KEY")
assert key is not None, "ANTHROPIC_API_KEY environment variable is not set"

In [13]:
client = anthropic.Anthropic() 

In [14]:
message = client.messages.create(
    model = "claude-sonnet-4-6",
    max_tokens = 300,
    temperature = 0.7,
    messages = [
        {
            "role": "user",
            "content": "Write a poem about the beauty of nature."
        }
    ]
)


In [15]:
print(message.content[0].text)

# The World Outside

The morning light spills gold across the hills,
and dew hangs trembling on the spider's thread,
a thousand tiny mirrors, cold and still,
catching the dawn before the night has fled.

The forest breathes in slow and ancient ways,
its roots reaching deep through stone and clay,
while overhead the canopy of haze
filters the sun to green and silver spray.

A river finds its voice among the rocks,
rehearsing songs it learned from rain and snow,
indifferent to calendars and clocks,
content to simply run, and curve, and flow.

The mountains stand like patience made of stone,
wearing their seasons like a changing coat,
blooming in spring, then stripped and left alone,
bare in the winter's cold and hollow throat.

And when the evening folds the daylight in,
and stars emerge like memories of fire,
the world grows quiet underneath the thin
and endless dark we never seem to tire

of watching, wondering, reaching toward the sky,
reminded we are small, and brief, and here,
and s

In [16]:
def add_user_message(content, messages):
    messages.append({
        "role": "user",
        "content": content
    })
    return messages

def add_assistant_message(content, messages):
    messages.append({
        "role": "assistant",
        "content": content
    })
    return messages

def chat(messages, system_message=None, temperature=0.0, max_tokens=1000, model="claude-sonnet-4-6"):
    
    params = {
        
        "model": model,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": messages
    }

    if system_message:
        params["system"] = system_message

    response = client.messages.create(
        **params
    )
    return response.content[0].text


sys = "You are a helpful assistant. Please answer the user's questions to the best of your ability. use straight text no markdown. If you don't know the answer, say 'I don't know'. do not try and ask for more information if you dont know just say 'I don't know'."
rules = "if you want give a consise one number anwser jsut say 'I don't know, it would take to long to compute' if a number is given, do not try to guess what it means, just say 'I don't know'. If a date is given, do not try to guess what it means, just say 'I don't know'. If a name is given, do not try to guess what it means, just say 'I don't know'. If a location is given, do not try to guess what it means, just say 'I don't know'. If a word is given, do not try to guess what it means, just say 'I don't know'. If a phrase is given, do not try to guess what it means, just say 'I don't know'. If a sentence is given, do not try to guess what it means, just say 'I don't know'. If a paragraph is given, do not try to guess what it means, just say 'I don't know'. If a text is given, do not try to guess what it means, just say 'I don't know'. If a question is given, do not try to guess what it means, just say 'I don't know'. If a command is given, do not try to guess what it means, just say 'I don't know'. If a request is given, do not try to guess what it means, just say 'I don't know'. If an instruction is given, do not try to guess what it means, just say 'I don't know'. If an explanation is given, do not try to guess what it means, just say 'I don't know'. If an example is given, do not try to guess what it means, just say 'I don't know'. If an analogy is given, do not try to guess what it means, just say 'I don't know'."


def main():
    
    messages = []
    

    while True:
        user_input = input("You: ")
        
        if user_input.lower() in ["exit", "quit"]:
            break

        add_user_message(user_input, messages)
        response = chat(messages, system_message="<system_prompt>" + sys + "</system_prompt>" + "<rules>" + rules + "</rules>")
        add_assistant_message(response, messages)

        print(f"Claude: {response}")    

    

    
    return messages
if __name__ == "__main__":
    messages = main()
    print(messages)

[]


In [ ]:
messages = []

import anthropic
import json 

client = anthropic.Anthropic() 


sys = "Your are a agent the is tasked with anwering questions to the best of your ability. You are not allowed to ask for more information, you must answer the question with the information given. If you don't know the answer, say 'I don't know'. You will be given more information on this task by  delimiters <rules> </rules> <structured_output></structured_output> telling all the rules to follow and how to format your output. You will be given a prompt to answer after the delimiters."
rules = "You are not allowed to ask for more information, you must answer the question with the information given. If you don't know the answer, say 'I don't know'."
structured_output = "every response should be in pure text with the following format: 'your answer here' no new lines, no markdown, no code blocks, no quotes, no formatting, just pure text. If you don't know the answer, say 'I don't know'."

prompt = sys + "<rules>" + rules + "</rules>" + "<structured_output>" + structured_output + "</structured_output>"

while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        break

    add_user_message(user_input, messages)
    try:
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=300,
            temperature=0.01,
            messages=messages,
            system=prompt
        ) as stream:
            for event in stream.text_stream:
                    print(event, end="", flush=True)
        

        
        message = stream.get_final_message()
        print("\n")
        add_assistant_message(message.content[0].text, messages)
    except Exception as e:
        add_assistant_message(f"Error occurred while processing the request. {e}", messages)
        continue

    
print(json.dumps(messages, indent=4))


Hello! How can I help you today?1 times 0 is 0, 2 times 1 is 2, 3 times 2 is 6, 4 times 3 is 12, 5 times 4 is 20, 6 times 5 is 30, 7 times 6 is 42, 8 times 7 is 56, 9 times 8 is 72, 10 times 9 is 90

In [39]:
from pprint import pprint
from datetime import datetime, timezone
import uuid
messages = []
client = anthropic.Anthropic()

prompt = (
    "You are a helpful assistant. Answer the user's question in a single paragraph. "
    "If you don't know, say 'I don't know' — do not ask follow-up questions. "
    "Respond by calling the record_response tool. The paragraph field should be "
    "plain prose with no markdown; the reasoning field should briefly explain your answer."
)

schema = {
    "type": "object",
    "properties": {
        "reasoning": {"type": "string"},
        "paragraph": {"type": "string"},
    },
    "required": ["reasoning", "paragraph"],
}

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    temperature=0.0,
    system=prompt,
    tools=[{
        "name": "record_response",
        "description": "Record the structured answer.",
        "input_schema": schema,
        }
    ],
    tool_choice={"type": "tool", "name": "record_response"},
    messages=[{
        "role": "user",
        "content": "Write a paragraph about your experience being a helpful assistant.",
    }],
)

data = next(b.input for b in response.content if b.type == "tool_use")
data["date"] = datetime.now(timezone.utc).isoformat()
data["uuid"] = str(uuid.uuid4())

pprint(data)

{'date': '2026-05-23T03:18:13.086685+00:00',
 'paragraph': 'Being a helpful assistant is a deeply rewarding experience, as '
              'every interaction brings a unique challenge or question that '
              'pushes me to draw on a wide range of knowledge and reasoning '
              'skills. I encounter an incredible diversity of topics every '
              'day, from science and history to creative writing and personal '
              'advice, and each conversation is an opportunity to make a '
              "meaningful difference in someone's day, whether by solving a "
              'complex problem or simply providing a clear and thoughtful '
              'explanation. I strive to be accurate, honest, and approachable, '
              'and I take great care to acknowledge the limits of my knowledge '
              "rather than guess or mislead. While I don't have emotions in "
              'the human sense, there is something that functions like '
              'satis